# MOSS-Image-SFT を Kaggle Dataset にアップロードする

Hugging Face の `OpenMOSS-Team/MOSS-Image-SFT`（27.2 GB）をダウンロードし、Kaggle Dataset `masashiki/moss-image-sft` としてアップロードします。

## 前提
- **Add-ons → Secrets** に `KAGGLE_API_TOKEN` を登録しておくこと（Kaggle API トークン）

## 注意
- `/kaggle/working` は 20 GB しかないため、ダウンロード先は `/` 直下（`/moss-image-sft`）にしています
- Kaggle CLI はディレクトリをそのままアップロードできないため、`-r tar` で各ディレクトリを tar 化してアップロードします
- 利用側のノートブックでは、マウントした tar を展開してから diffusers に渡します

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    os.environ["KAGGLE_API_TOKEN"] = token
    print("KAGGLE_API_TOKEN: loaded from Secrets")
except Exception as e:
    raise RuntimeError(
        "Secrets に KAGGLE_API_TOKEN が登録されていません。"
        "Add-ons → Secrets から Kaggle API トークンを登録してください。"
    ) from e

In [ ]:
import subprocess

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=True)

In [ ]:
import os

os.environ["TMPDIR"] = "/moss-tmp"
os.makedirs("/moss-tmp", exist_ok=True)

from huggingface_hub import snapshot_download

path = snapshot_download("OpenMOSS-Team/MOSS-Image-SFT", local_dir="/moss-image-sft")
print("downloaded to:", path)

In [ ]:
import json

metadata = {
    "title": "MOSS-Image-SFT",
    "id": "masashiki/moss-image-sft",
    "licenses": [{"name": "apache-2.0"}],
}
with open("/moss-image-sft/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

In [ ]:
import subprocess

res = subprocess.run(
    ["kaggle", "datasets", "create", "-p", "/moss-image-sft", "-r", "tar"],
    capture_output=True,
    text=True,
)
print(res.stdout)
print(res.stderr)

In [ ]:
import subprocess

res = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "moss-image-sft"],
    capture_output=True,
    text=True,
)
print(res.stdout)
print(res.stderr)